Data loading and exploration
Importing the dataset.
Checking dimensions.
Inspecting structure and data types.
Viewing the first few observations.
Checking for missing values.
Descriptive statistics.

In [14]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
import joblib
import json
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_val_score, KFold






df=pd.read_csv("../data/Clean_Dataset.csv")
# print(df)
# print(df.ndim)
# print(df.dtypes)
# print(df.head())
# print(df.columns)
# print(df[df.isnull()])
# print(df.describe())
# print(df['Unnamed: 0'])

Classification of variables by nature:
categorical variables;
ordinal categorical variables;
quantitative variables;
target variable.

In [29]:
# nominales =pd.Categorical(df)
# ordinales =pd.Categorical(df,ordered=True)
# print(categorie)
# quantitative_discrete =df.select_dtypes(include='integer')
# quantitative_continuous =df.select_dtypes(include='float')
# quantitative_discrete=[]
# quantitative_continuous=[]
quantitative_discrete=[
                       'airline','flight',
                       'source_city','departure_time',
                       'stops','arrival_time',
                       'destination_city','class',
                       'days_left'
                       ]

quantitative_continuous=['duration','price']



# print(quantitative_discrete)
# print(quantitative_continuous)
# print(f"ordinales : {ordinales}")
# print(f"nominales : {nominales}")

v_cible=df["price"]




3️⃣ Univariate analysis
Categorical variables
Analysis of:

frequencies;
distributions;
dominant categories;
potential imbalances.
Use of appropriate visualizations.

Quantitative variables
Analysis of:

descriptive statistics;
distributions;
dispersion;
extreme values;
potential skewness.
Use of histograms and boxplots, in particular.

In [74]:
def discrete_frequency_graph(value):
    print(df[value].value_counts())
    print(df[value].describe())
    df[value].value_counts().plot(kind="bar")
    plt.xlabel(value)
    plt.ylabel("Number of tickets")
    plt.title(value)
    plt.show()

def continuous_frequency_graph(value):
    print(df[value].describe())
    df[value].plot(kind="hist", bins=30)
    plt.xlabel(value)
    plt.ylabel("Frequency")
    plt.title(value)
    plt.show()



In [77]:
descriptive_statistics=df['price'].describe()
print(descriptive_statistics)


# 
def valeur_extreme_continuous(value):
    Q1 = df[f"{value}"].quantile(0.25)
    Q3 = df[f"{value}"].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    valeur_extreme = df[(df[f"{value}"] < lower) | (df[f"{value}"] > upper)]

    potential_asymmetry=df['price'].skew()
    print(f"valeur extreme : {valeur_extreme}")
    print(f"potential asymmetry : {potential_asymmetry}")
    continuous_frequency_graph(f"{value}")
    return valeur_extreme,potential_asymmetry


# plt.boxplot(df["price"].dropna())

# plt.ylabel("Price")
# plt.title("Boxplot of Ticket Prices")

# plt.show()

count    300153.000000
mean      20889.660523
std       22697.767366
min        1105.000000
25%        4783.000000
50%        7425.000000
75%       42521.000000
max      123071.000000
Name: price, dtype: float64


durée du vol et prix ;

compagnie aérienne et prix ;

nombre d’escales et prix ;

nombre de jours avant le vol et prix.

In [78]:

def Analyse_multivariee(value1,value2):
    if(value1 in quantitative_continuous and value2 in quantitative_continuous):
           plt.bar(df[f'{value1}'], df[f'{value2}'])
           plt.xlabel(f"{value1}")
           plt.ylabel(f"{value2}")
           plt.title(f"{value1} with {value2}")
           plt.show()
    elif(value2 in quantitative_discrete and value1 in quantitative_discrete):
            pd.crosstab(df[f'{value1}'], df[f'{value2}']).plot(kind='bar')
            plt.title('airline vs stops')
            plt.xlabel('airline')
            plt.ylabel('count')
            plt.show()
    else:
        sns.boxplot(x=df[f"{value1}"], y=df[f"{value2}"])
        plt.xlabel(f"{value1}")
        plt.ylabel(f"{value2}")
        plt.title(f"{value1} with {value2}")
        plt.show()

In [ ]:
Analyse_multivariee('duration','price')

In [ ]:
Analyse_multivariee('airline','price')

In [ ]:
Analyse_multivariee('stops','price')

In [ ]:
Analyse_multivariee('days_left','price')

flight duration and price;

airline and price;

number of stopovers and price;

number of days before the flight and price.

In [3]:


ord_encoder = OrdinalEncoder()
ohe = OneHotEncoder(sparse_output=False)

def Preprocessing(data, split=True):
    df = data.dropna().copy()
    ordinales = ['class', 'stops']
    nominales = ['airline', 'source_city', 'departure_time', 'arrival_time', 'destination_city']

    if split:
        ord_encoder.fit(df[ordinales])
        ohe.fit(df[nominales])

    df[ordinales] = ord_encoder.transform(df[ordinales])
    encoded = ohe.transform(df[nominales])

    df = df.drop(columns=nominales + ['flight', 'Unnamed: 0'], errors='ignore')
    df[ohe.get_feature_names_out(nominales)] = encoded

    X = df.drop(columns=['price'], errors='ignore')
    y = df['price'] if 'price' in df.columns else None

    if not split:
        return X, None, None, None, df

    X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.20, shuffle=True)

    print(f"X_train : {X_train.shape}")
    print(f"y_train : {y_train.shape}")
    print(f"X_test : {X_test.shape}")
    print(f"y_test : {y_test.shape}")

    return X_train, X_test, y_train, y_test, df

X_train, X_test, y_train, y_test, df=Preprocessing(df)

X_train : (240122, 34)
y_train : (240122,)
X_test : (60031, 34)
y_test : (60031,)


handling missing values;

transforming ordinal variables;

encoding categorical variables;

selecting the variables to be used;

separating explanatory variables from the target variable;

splitting the data into training and test sets.

In [31]:


def Linear_Regression():
    regr = LinearRegression()
    pipeline = Pipeline([
                ("model",regr)
            ])
    
    regr.fit(X_train, y_train) 
    print(regr.score(X_test, y_test))

    y_pred = regr.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    mse=mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)

    print("R²:", regr.score(X_test, y_test))
    print("MAE:", mae)
    print("MSE:", mse)
    print("RMSE:", rmse)
    return pipeline
   

Linear_Regression()


0.9090219980252148
R²: 0.9090219980252148
MAE: 4578.472236219375
MSE: 46897547.30271565
RMSE: 6848.1783930265465


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](34,)","['stops','class','duration',...,'destination_city_Hyderabad', 'destination_city_Kolkata','destination_city_Mumbai']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,34
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None


In [4]:


def Random_Forest_Regressor():
    model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    pipeline = Pipeline([
            
            ("model",model)
            
        ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    num_folds = 5
    kf = KFold(n_splits=num_folds, shuffle=True, random_state=42)
    scores = cross_val_score(
    model,
    X_train,
    y_train,
    cv=kf,
    scoring="r2"  
)

    print("Scores:", scores)

    print(f"R²   : {r2_score(y_test, y_pred):.4f}")
    print(f"MAE  : {mean_absolute_error(y_test, y_pred):.2f}")
    print(f"MSE  : {mean_squared_error(y_test, y_pred):.2f}")
    print(f"RMSE : {root_mean_squared_error(y_test, y_pred):.2f}")
    return pipeline

Random_Forest_Regressor()

    




Scores: [0.98520735 0.98532477 0.98507883 0.98508459 0.98494312]
R²   : 0.9849
MAE  : 1077.50
MSE  : 7763899.27
RMSE : 2786.38


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](34,)","['stops','class','duration',...,'destination_city_Hyderabad', 'destination_city_Kolkata','destination_city_Mumbai']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,34
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterio

In [72]:
print(df)

        stops  class  duration  days_left  price  airline_AirAsia  \
0         2.0    1.0      2.17          1   5953              0.0   
1         2.0    1.0      2.33          1   5953              0.0   
2         2.0    1.0      2.17          1   5956              1.0   
3         2.0    1.0      2.25          1   5955              0.0   
4         2.0    1.0      2.33          1   5955              0.0   
...       ...    ...       ...        ...    ...              ...   
300148    0.0    0.0     10.08         49  69265              0.0   
300149    0.0    0.0     10.42         49  77105              0.0   
300150    0.0    0.0     13.83         49  79099              0.0   
300151    0.0    0.0     10.00         49  81585              0.0   
300152    0.0    0.0     10.08         49  81585              0.0   

        airline_Air_India  airline_GO_FIRST  airline_Indigo  airline_SpiceJet  \
0                     0.0               0.0             0.0               1.0   
1        

In [15]:
def Export():

    pipeline=Random_Forest_Regressor()
    joblib.dump(pipeline, "../artifacts/flight_price_model.joblib")


    metadata = {
        "model_name": "RandomForestRegressor",
        "model_file": "flight_price_model.joblib",
        "target": "price",
        "features": list(X_train.columns),
        "n_features": X_train.shape[1],
        "metrics": {
            "R2": 0.9849,
            "MAE": 1076.74,
            "MSE"  : 7790497.92,
            "RMSE": 2791.15
        }
    }

    with open("../artifacts/flight_price_model.meta.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=4)

Export()

Scores: [0.98520735 0.98532477 0.98507883 0.98508459 0.98494312]
R²   : 0.9849
MAE  : 1077.50
MSE  : 7763899.27
RMSE : 2786.38


In [20]:
import pandas as pd
import joblib
from pathlib import Path

sample = pd.DataFrame([{
    "airline": "Air_India",
    "source_city": "Delhi",
    "destination_city": "Mumbai",
    "departure_time": "Early_Morning",
    "arrival_time": "Afternoon",
    "stops": "one",
    "class": "Economy",
    "duration": 8.83,
    "days_left": 1
}])

# Preprocess
X_sample, _, _, _, _ = Preprocessing(sample, split=False)

# Load model
model_path = Path().resolve().parent / "artifacts" / "flight_price_model.joblib"
pipeline = joblib.load(model_path)

price = pipeline.predict(X_sample)
print(f"Predicted price is: {price[0]:,.0f} INR")

Predicted price is: 14,254 INR
